In [0]:
WITH params AS (
  SELECT 30 AS p_ml_window_days
),
runs_scoped AS (
  SELECT
    CAST(r.run_id AS STRING) AS run_id,
    CAST(r.workspace_id AS STRING) AS workspace_id,
    CAST(r.experiment_id AS STRING) AS experiment_id,
    r.run_name,
    r.status,
    r.start_time AS run_start_time_utc,
    r.end_time AS run_end_time_utc
  FROM system.mlflow.runs_latest r
  CROSS JOIN params p
  WHERE r.delete_time IS NULL
    AND r.start_time >= timestampadd(DAY, -p.p_ml_window_days, current_timestamp())
),
metrics_scoped AS (
  SELECT
    CAST(h.run_id AS STRING) AS run_id,
    lower(h.metric_name) AS metric_name_lc,
    CAST(h.metric_value AS DOUBLE) AS metric_value,
    h.metric_time
  FROM system.mlflow.run_metrics_history h
  CROSS JOIN params p
  INNER JOIN runs_scoped r
    ON CAST(h.run_id AS STRING) = r.run_id
  WHERE h.metric_time >= timestampadd(DAY, -p.p_ml_window_days, current_timestamp())
),
training_metric_candidates AS (
  SELECT
    run_id,
    metric_name_lc AS training_data_count_key,
    metric_value AS training_data_count,
    metric_time AS training_data_count_observed_at_utc,
    CASE
      WHEN metric_name_lc IN (
        'training_example_count',
        'train_example_count',
        'training_sample_count',
        'train_sample_count',
        'training_row_count',
        'train_row_count',
        'training_rows',
        'train_rows',
        'dataset_row_count',
        'dataset_rows',
        'num_train_rows',
        'num_train_samples'
      ) THEN 220
      WHEN metric_name_lc RLIKE '(train|training|dataset).*(example|sample|row|record).*(count|num|size)' THEN 170
      WHEN metric_name_lc RLIKE '(example|sample|row|record).*(count|num|size).*(train|training|dataset)' THEN 160
      WHEN metric_name_lc RLIKE '(train|training|dataset).*(example|sample|row|record)' THEN 130
      ELSE 70
    END AS candidate_score
  FROM metrics_scoped
  WHERE metric_value IS NOT NULL
    AND NOT isnan(metric_value)
    AND metric_value > 0
    AND metric_value < 1000000000000000
    AND metric_name_lc RLIKE '(train|training|dataset|example|examples|sample|samples|row|rows|record|records)'
),
training_metric_best AS (
  SELECT
    run_id,
    training_data_count,
    training_data_count_key,
    training_data_count_observed_at_utc
  FROM (
    SELECT
      run_id,
      training_data_count,
      training_data_count_key,
      training_data_count_observed_at_utc,
      candidate_score,
      ROW_NUMBER() OVER (
        PARTITION BY run_id
        ORDER BY candidate_score DESC, training_data_count_observed_at_utc DESC, training_data_count DESC
      ) AS rn
    FROM training_metric_candidates
    WHERE training_data_count IS NOT NULL
  ) ranked
  WHERE rn = 1
),
runs_enriched AS (
  SELECT
    r.workspace_id,
    r.experiment_id,
    r.run_id,
    r.run_name,
    r.status,
    r.run_start_time_utc,
    r.run_end_time_utc,
    t.training_data_count,
    t.training_data_count_key,
    t.training_data_count_observed_at_utc
  FROM runs_scoped r
  LEFT JOIN training_metric_best t
    ON r.run_id = t.run_id
),
summary AS (
  SELECT
    COUNT(*) AS total_runs_30d,
    SUM(CASE WHEN training_data_count IS NOT NULL THEN 1 ELSE 0 END) AS runs_with_training_data_count,
    MAX(training_data_count) AS max_training_data_count,
    MIN(training_data_count) AS min_training_data_count,
    ROUND(AVG(training_data_count), 2) AS avg_training_data_count
  FROM runs_enriched
),
latest_with_count AS (
  SELECT
    workspace_id,
    experiment_id,
    run_id,
    run_name,
    status,
    training_data_count,
    training_data_count_key,
    run_start_time_utc,
    run_end_time_utc,
    training_data_count_observed_at_utc
  FROM (
    SELECT
      workspace_id,
      experiment_id,
      run_id,
      run_name,
      status,
      training_data_count,
      training_data_count_key,
      run_start_time_utc,
      run_end_time_utc,
      training_data_count_observed_at_utc,
      ROW_NUMBER() OVER (
        ORDER BY COALESCE(run_end_time_utc, run_start_time_utc) DESC, run_id DESC
      ) AS rn
    FROM runs_enriched
    WHERE training_data_count IS NOT NULL
  ) ranked
  WHERE rn = 1
),
metric_key_profile AS (
  SELECT
    concat_ws(', ', slice(array_sort(collect_set(training_data_count_key)), 1, 50)) AS metric_like_keys_30d
  FROM training_metric_candidates
)
SELECT
  s.total_runs_30d,
  s.runs_with_training_data_count,
  s.max_training_data_count,
  s.min_training_data_count,
  s.avg_training_data_count,
  l.workspace_id AS latest_workspace_id,
  l.experiment_id AS latest_experiment_id,
  l.run_id AS latest_run_id,
  l.run_name AS latest_run_name,
  l.status AS latest_run_status,
  l.training_data_count AS latest_training_data_count,
  l.training_data_count_key AS latest_training_data_key,
  l.run_start_time_utc AS latest_run_start_time_utc,
  from_utc_timestamp(l.run_start_time_utc, 'Asia/Seoul') AS latest_run_start_time_kst,
  l.run_end_time_utc AS latest_run_end_time_utc,
  from_utc_timestamp(l.run_end_time_utc, 'Asia/Seoul') AS latest_run_end_time_kst,
  l.training_data_count_observed_at_utc AS latest_metric_time_utc,
  from_utc_timestamp(l.training_data_count_observed_at_utc, 'Asia/Seoul') AS latest_metric_time_kst,
  mkp.metric_like_keys_30d,
  current_timestamp() AS generated_at_utc,
  from_utc_timestamp(current_timestamp(), 'Asia/Seoul') AS generated_at_kst
FROM summary s
LEFT JOIN latest_with_count l ON TRUE
LEFT JOIN metric_key_profile mkp ON TRUE;